In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader

In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
batch_size = 16
lr = 5e-5
epochs = 1
temperature = 2.0
alpha_soft = 0.5
max_len = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# Load dataset
raw = load_dataset("tweet_eval", "sentiment")

README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [5]:
raw

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [6]:
label_features = raw["train"].features["label"]

In [7]:
print(f"Label Names: {label_features.names}")

Label Names: ['negative', 'neutral', 'positive']


In [8]:
# Training subset
train = raw["train"].shuffle(seed = 42).select(range(2500))

In [9]:
val = raw["validation"]

In [10]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [11]:
def tokenize(example):
  return tokenizer(example["text"], truncation=True, max_length = max_len)

In [12]:
# Tokenize & remove original text column
tokenized = {}

tokenized["train"] = train.map(tokenize, batched = True, remove_columns = ["text"])
tokenized["validation"] = val.map(tokenize, batched = True, remove_columns = ["text"])

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [13]:
tokenized

{'train': Dataset({
     features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 2500
 }),
 'validation': Dataset({
     features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 2000
 })}

In [14]:
# Data Collator (auto-padding)
collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

In [15]:
## DataLoaders
train_loader = DataLoader(tokenized['train'], batch_size = batch_size, shuffle = True, collate_fn = collator)
val_loader = DataLoader(tokenized["validation"], batch_size = batch_size, shuffle = True, collate_fn = collator)

### **`Load Teacher & Student Models`**

In [16]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

num_labels = 3

teacher = AutoModelForSequenceClassification.from_pretrained("bert-large-uncased", num_labels = num_labels).to(device)
student = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = num_labels).to(device)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
# Freeze teacher (no training)
for p in teacher.parameters():
  p.requires_grad = False

In [18]:
!pip install torchinfo

In [19]:
import torchinfo
torchinfo.summary(teacher)

Layer (type:depth-idx)                                       Param #
BertForSequenceClassification                                --
├─BertModel: 1-1                                             --
│    └─BertEmbeddings: 2-1                                   --
│    │    └─Embedding: 3-1                                   (31,254,528)
│    │    └─Embedding: 3-2                                   (524,288)
│    │    └─Embedding: 3-3                                   (2,048)
│    │    └─LayerNorm: 3-4                                   (2,048)
│    │    └─Dropout: 3-5                                     --
│    └─BertEncoder: 2-2                                      --
│    │    └─ModuleList: 3-6                                  (302,309,376)
│    └─BertPooler: 2-3                                       --
│    │    └─Linear: 3-7                                      (1,049,600)
│    │    └─Tanh: 3-8                                        --
├─Dropout: 1-2                                      

In [20]:
teacher.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1

In [21]:
ce_loss = nn.CrossEntropyLoss()

In [22]:
kl_loss = nn.KLDivLoss(reduction = 'batchmean')

In [23]:
optimizer = optim.AdamW(student.parameters(), lr = lr)

In [24]:
from transformers import get_scheduler
lr_schedular = get_scheduler(
    name = "linear",
    optimizer = optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_loader) * epochs
)

In [25]:
from tqdm.auto import tqdm

In [26]:
def distil_epoch():
  student.train()
  pbar = tqdm(train_loader, desc = 'Train')
  for batch in pbar:
    input_ids = batch["input_ids"].to(device)
    attention = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # Teacher Predictions (soft labels)
    with torch.no_grad():
      t_logits = teacher(input_ids, attention_mask = attention).logits
      t_soft = torch.softmax(t_logits / temperature, dim = 1)

    # Student Predictions
    s_logits = student(input_ids, attention_mask = attention).logits
    s_soft = torch.log_softmax(s_logits / temperature, dim = 1)

    # Distillation + ce loss
    loss_soft = kl_loss(s_soft, t_soft) * (temperature ** 2)
    loss_hard = ce_loss(s_logits, labels)
    loss = alpha_soft * loss_soft + (1 - alpha_soft) * loss_hard

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    lr_schedular.step()
    pbar.set_postfix({"loss": f"{loss.item():.4f}"})

In [27]:
def evaluate():
  student.eval()
  correct = total = 0
  with torch.no_grad():
    for batch in val_loader:
      ids = batch["input_ids"].to(device)
      attention = batch["attention_mask"].to(device)
      lbl = batch["labels"].to(device)
      out = student(ids, attention_mask = attention).logits
      pred = out.argmax(dim = 1)
      correct += (pred == lbl).sum().item()
      total += lbl.size(0)
  return round(correct / total * 100, 2)

In [28]:
for ep in range(1, epochs + 1):
  distil_epoch()
  acc = evaluate()
  print(f"Epoch: {ep}/{epochs} | Validation Accuracy: {acc}%")

Train:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch: 1/1 | Validation Accuracy: 65.45%


In [29]:
# Save the model
student.save_pretrained("distilled_student_model")
tokenizer.save_pretrained("distilled_student_tokenizer")

('distilled_student_tokenizer/tokenizer_config.json',
 'distilled_student_tokenizer/special_tokens_map.json',
 'distilled_student_tokenizer/vocab.txt',
 'distilled_student_tokenizer/added_tokens.json',
 'distilled_student_tokenizer/tokenizer.json')

In [30]:
# Load test set
test = load_dataset("tweet_eval", "sentiment", split = "test[:500]")

In [31]:
tokenized_test = test.map(tokenize, batched = True, remove_columns = ["text"])
tokenized_test.set_format(type = 'torch', columns = ["input_ids", "attention_mask", "label"])
test_loader = DataLoader(tokenized_test, batch_size = batch_size, shuffle = True, collate_fn = collator)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [32]:
from sklearn.metrics import accuracy_score
import time

def predict_and_evaluate(model, name, test_loader):
  model.eval()
  all_preds, all_labels = [], []
  start_time = time.time()

  with torch.no_grad():
    for batch in test_loader:
      ids = batch["input_ids"].to(device)
      attention = batch["attention_mask"].to(device)
      lbls = batch["labels"].to(device)

      logits = model(ids, attention_mask = attention).logits
      preds = torch.argmax(logits, dim = 1)

      all_preds.extend(preds.cpu().tolist())
      all_labels.extend(lbls.cpu().tolist())
  total_time = time.time() - start_time
  acc = accuracy_score(all_labels, all_preds)
  avg_time = total_time / len(test_loader.dataset)

  print(f"\n {name}")
  print(f"Accuracy: {acc * 100:.2f}%")
  print(f"Total Inference Time: {total_time:.2f} sec")
  print(f"Avg Time per sample: {avg_time:.2f} sec")
  return acc, total_time, avg_time

In [33]:
# Compare teacher vs student
predict_and_evaluate(teacher, name = "Teacher (BERT-Large)", test_loader=test_loader)
predict_and_evaluate(student, name = "Student (BERT-Base)", test_loader=test_loader)


 Teacher (BERT-Large)
Accuracy: 26.60%
Total Inference Time: 4.47 sec
Avg Time per sample: 0.01 sec

 Student (BERT-Base)
Accuracy: 58.00%
Total Inference Time: 1.52 sec
Avg Time per sample: 0.00 sec


(0.58, 1.5221376419067383, 0.0030442752838134766)